# Curanex: DME from a retinal OCT scan - threshold set on OCTDL, tested on Duke (CPU)

The Kermany-trained OCT model ranked DME well on OCTDL (AUC 0.979) but its validation threshold caught only 62%.
**Rules fixed before running (2026-11-12):** model unchanged; DME score = P(DME); threshold = highest with sensitivity >= 0.85 on
OCTDL DME vs normal (OCTDL already used once, so it is now the calibration set). **Outside test, scored once:** Duke / Srinivasan
2014 (USA), 15 DME vs 15 normal eyes, one score per eye = 90th percentile of its B-scan DME scores (the same eye rule as the AMD
check); B-scan level reported too. Verified if eye-level sensitivity and AUC are both >= 0.80; live at >= 0.75; otherwise off.

In [ ]:
!pip install -q onnxruntime

In [ ]:
import glob, json, os, re
import numpy as np, onnxruntime as ort
from PIL import Image
from sklearn.metrics import roc_auc_score
meta = json.load(open(glob.glob("/kaggle/input/**/oct_ft.json", recursive=True)[0]))
sess = ort.InferenceSession(glob.glob("/kaggle/input/**/oct_ft.onnx", recursive=True)[0])
MEAN, STD = np.array(meta["mean"], np.float32), np.array(meta["std"], np.float32)


def prep(f):  # Resize(256) + CenterCrop(224) + normalise, as in training and in the app
    im = Image.open(f).convert("RGB"); w, h = im.size; s = 256 / min(w, h)
    im = im.resize((max(256, round(w * s)), max(256, round(h * s))), Image.BILINEAR); w, h = im.size
    l, t = (w - 224) // 2, (h - 224) // 2
    a = (np.asarray(im.crop((l, t, l + 224, t + 224)), np.float32) / 255 - MEAN) / STD
    return a.transpose(2, 0, 1)


def report(y, s, t):
    y, s = np.array(y), np.array(s); fl = s >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(fl[y == 1].mean()), 3),
         "specificity": round(float((~fl[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, s)), 3)}
    low = min(r["sensitivity"], r["auc"]); r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r




def scores(files):
    out = []
    for i in range(0, len(files), 64):
        p = sess.run(None, {"image": np.stack([prep(f) for f in files[i:i + 64]])})[0]
        out += list(p[:, 1])
    return np.array(out)


img = lambda f: f.lower().endswith((".jpg", ".jpeg", ".png", ".tif"))
folder = lambda f: os.path.basename(os.path.dirname(f)).upper()
octdl = [f for f in glob.glob("/kaggle/input/**/*", recursive=True) if img(f) and "duke" not in f.lower() and folder(f) in ("DME", "NO")]
duke = [f for f in glob.glob("/kaggle/input/**/*", recursive=True) if img(f) and "duke" in f.lower() and folder(f) in ("DME", "NORMAL")]
print("OCTDL:", {k: sum(folder(f) == k for f in octdl) for k in ("DME", "NO")}, "| Duke:", {k: sum(folder(f) == k for f in duke) for k in ("DME", "NORMAL")})
assert octdl and duke
pc = scores(octdl); yc = np.array([folder(f) == "DME" for f in octdl], int)
T = float(max(v for v in np.unique(np.round(pc, 6)) if (pc[yc == 1] >= v).mean() >= 0.85))
print("threshold", T, "calibration (OCTDL):", report(yc, pc, T))
pd_ = scores(duke)
eyes = {}
for f, s in zip(duke, pd_):
    eyes.setdefault(re.split(r"[_ ]", os.path.basename(f))[0], []).append(s)
ey = [1 if k.upper().startswith("DME") else 0 for k in eyes]
es = [float(np.percentile(v, 90)) for v in eyes.values()]
res = {"threshold": T, "eye_level": report(ey, es, T), "bscan_level": report([int(folder(f) == "DME") for f in duke], pd_, T)}
print("OUTSIDE TEST on Duke (once):", res)
json.dump(res, open("/kaggle/working/dme_duke.json", "w"), indent=1)